NOTEBOOK 6: Random Forest Classifier
Run AFTER Notebook 5 is complete.
 
Inputs:
    - analytical_dataset.csv
    - ticker_sectors.csv          (from Notebook 5)
 
Outputs:
    - rf_predictions.csv          (test set predictions + probabilities)
    - rf_feature_importance.csv   (ranked feature importances)
    - rf_results_summary.txt      (report-ready summary)
 
Key design decisions:
    - BUY trades only (sell trades excluded — Y label is not meaningful for sells)
    - Spark MLlib for Random Forest training
    - 5-fold CV for hyperparameter tuning
    - AUC-ROC as primary metric
"""

# CELL 1 — Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
import os
from pathlib import Path

warnings.filterwarnings('ignore')

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, StringType
from pyspark.ml import Pipeline
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.feature import (
    VectorAssembler,
    StringIndexer,
    OneHotEncoder,
    StandardScaler as SparkScaler,
)
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder, TrainValidationSplit

# For confusion matrix and additional metrics (sklearn on test predictions)
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_curve,
    auc,
    ConfusionMatrixDisplay,
)

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update(
    {
        'figure.dpi': 120,
        'axes.titlesize': 13,
        'axes.labelsize': 11,
        'xtick.labelsize': 9,
        'ytick.labelsize': 9,
    }
)

# CONFIG
DATA_DIR = Path("data")
OUTPUT_DIR = DATA_DIR / "output"

ANALYTICAL_PATH = OUTPUT_DIR / "analytical_dataset.csv"
SECTORS_PATH = OUTPUT_DIR / "ticker_sectors.csv"
PREDICTIONS_OUT = OUTPUT_DIR / "rf_predictions.csv"
IMPORTANCE_OUT = OUTPUT_DIR / "rf_feature_importance.csv"
SUMMARY_OUT = OUTPUT_DIR / "rf_results_summary.txt"

RANDOM_SEED = 42
TEST_SPLIT = 0.2  # 80/20 train/test
CV_FOLDS = 5

# Committee proxy lookup
# Based on publicly documented committee assignments
# chamber = Senate/House used as fallback
COMMITTEE_LOOKUP = {
    # Key politicians with known committee assignments
    # Format: 'politician name': 'committee'
    'Nancy Pelosi': 'Science and Technology',
    'Ro Khanna': 'Science and Technology',
    'Dan Crenshaw': 'Armed Services',
    'Michael McCaul': 'Foreign Affairs',
    'Adam Schiff': 'Finance',
    'Shelley Moore Capito': 'Energy and Commerce',
    'John Boozman': 'Agriculture',
    'Angus King': 'Armed Services',
    'Tommy Tuberville': 'Armed Services',
    'Marjorie Taylor Greene': 'Science and Technology',
}

# Committee -> ordinal encoding
COMMITTEE_ORDINAL = {
    'Armed Services': 1,
    'Financial Services': 2,
    'Energy and Commerce': 3,
    'Science and Technology': 4,
    'Agriculture': 5,
    'Transportation': 6,
    'Foreign Affairs': 7,
    'Intelligence': 8,
    'Finance': 9,
    'Banking': 10,
    'Commerce': 11,
    'Health': 12,
    'Environment': 13,
    'Other': 0,
}

print("All imports loaded successfully")
print(f"Analytical path: {ANALYTICAL_PATH}")

All imports loaded successfully
Analytical path: data/output/analytical_dataset.csv


# CELL 2 — Start Spark & Load Data

Run this cell below if Cell 2 fails. Or to play safe, just run this cell together.

In [ ]:
import os

# 1. Update package lists so it can find the packages
!sudo apt-get update -qq

# 2. Install Java 17 instead of 11
!sudo apt-get install openjdk-17-jdk-headless -qq > /dev/null

# 3. Set the JAVA_HOME environment variable to the Java 17 path
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

In [ ]:
spark = SparkSession.builder \
    .appName("SC2320_Group6_Project_RandomForest") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()
 
spark.sparkContext.setLogLevel("ERROR")
print(f"Spark version: {spark.version}")
 
# Load analytical dataset
analytical_pdf = pd.read_csv(ANALYTICAL_PATH, parse_dates=['trade_date'])
sectors_pdf    = pd.read_csv(SECTORS_PATH)
 
print(f"Analytical dataset : {len(analytical_pdf):,} rows")
print(f"Sectors dataset    : {len(sectors_pdf):,} tickers")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/22 11:16:29 WARN Utils: Your hostname, codespaces-a752f8, resolves to a loopback address: 127.0.0.1; using 10.0.1.95 instead (on interface eth0)
26/04/22 11:16:29 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/22 11:16:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.1.1
Python executable: /usr/local/python/3.12.1/bin/python
Analytical dataset : 22,828 rows
Sectors dataset    : 1,521 tickers


# CELL 3 — Filter to BUY Trades & Valid Y Labels

In [ ]:
print("=" * 60)
print("FILTERING DATASET")
print("=" * 60)
 
step_counts = {'Total rows': len(analytical_pdf)}
 
#BUY trades only

buys = analytical_pdf[analytical_pdf['trade_type_encoded'] == 1.0].copy()
step_counts['BUY trades only'] = len(buys)
 
#Drop rows where Y label is NaN
# (trades too recent — no 60-day forward window available)
buys = buys.dropna(subset=['Y'])
step_counts['Valid Y label'] = len(buys)
 

#Drop blind spot > 60 days (leakage prevention)
buys = buys[buys['filed_after_days'] <= 60]
step_counts['Blind spot <= 60 days (no leakage)'] = len(buys)

#Drop rows with NaN in key features
key_features = ['size_bracket_ordinal', 'filed_after_days',
                'sigma_30', 'politician_rolling_abn_ret',
                'abn_ret_mean', 'anomaly_flag_L1']
buys = buys.dropna(subset=key_features)
step_counts['Valid features'] = len(buys)
 
# Print funnel
print(f"\n{'Step':<30} {'Rows':>8} {'Dropped':>10}")
print("-" * 50)
prev = None
for step, count in step_counts.items():
    dropped = f"-{prev - count:,}" if prev is not None else ""
    print(f"{step:<30} {count:>8,} {dropped:>10}")
    prev = count
 
print(f"\nClass balance:")
print(f"  Y=1 (beat SPY) : {(buys['Y']==1).sum():,} "
      f"({(buys['Y']==1).mean()*100:.1f}%)")
print(f"  Y=0 (missed)   : {(buys['Y']==0).sum():,} "
      f"({(buys['Y']==0).mean()*100:.1f}%)")
 

FILTERING DATASET

Step                               Rows    Dropped
--------------------------------------------------
Total rows                       22,828           
BUY trades only                  10,343    -12,485
Valid Y label                     9,434       -909
Blind spot <= 60 days (no leakage)    9,434         -0
Valid features                    9,434         -0

Class balance:
  Y=1 (beat SPY) : 4,262 (45.2%)
  Y=0 (missed)   : 5,172 (54.8%)


# CELL 4 — Feature Engineering

In [ ]:
print("=" * 60)
print("FEATURE ENGINEERING")
print("=" * 60)
 
df = buys.copy()
 
#  Feature 1: Merge sector
df = df.merge(sectors_pdf[['ticker', 'sector']], on='ticker', how='left')
df['sector'] = df['sector'].fillna('Other')
print(f"Sector nulls after merge: {df['sector'].isna().sum()}")
 
# Feature 2: Committee proxy
df['committee'] = df['politician'].map(COMMITTEE_LOOKUP).fillna('Other')
df['committee_ordinal'] = df['committee'].map(COMMITTEE_ORDINAL).fillna(0)
print(f"Committee distribution:")
print(df['committee'].value_counts().to_string())
 
# Feature 3: Trade year & quarter (market regime) 
df['trade_year']    = pd.to_datetime(df['trade_date']).dt.year
df['trade_quarter'] = pd.to_datetime(df['trade_date']).dt.quarter
 
# Feature 4: Blind spot intensity score 
# Combines anomaly magnitude with window length
# Longer windows with more anomalous days = higher intensity
df['blind_intensity'] = (
    df['abn_ret_mean'] * df['n_blind_spot_days'] / 
    df['filed_after_days'].clip(lower=1)
)
 
# Feature 5: Owner encoded 
owner_map = {'Self': 3, 'Joint': 2, 'Spouse': 1, 
             'Child': 0, 'Undisclosed': 0}
df['owner_encoded'] = df['owner'].map(owner_map).fillna(0)
 
print(f"\nFinal feature set shape: {df.shape}")
print(f"Unique sectors: {df['sector'].nunique()}")

FEATURE ENGINEERING
Sector nulls after merge: 0
Committee distribution:
committee
Science and Technology    4491
Other                     3914
Foreign Affairs            822
Armed Services             114
Agriculture                 66
Energy and Commerce         27

Final feature set shape: (9434, 44)
Unique sectors: 12


# CELL 5 — Load Into Spark & Assemble Features

In [ ]:
# Convert to Spark DataFrame
sdf = spark.createDataFrame(df)
 
print(f"Spark DataFrame created: {sdf.count():,} rows")
 
# String indexers for categorical features
# Spark MLlib requires categorical strings to be indexed first
 
sector_indexer   = StringIndexer(
    inputCol='sector', outputCol='sector_idx',
    handleInvalid='keep'
)
chamber_indexer  = StringIndexer(
    inputCol='chamber', outputCol='chamber_idx',
    handleInvalid='keep'
)
party_indexer    = StringIndexer(
    inputCol='party', outputCol='party_idx',
    handleInvalid='keep'
)
 
#One-hot encode sector (11 categories → binary vector)
sector_encoder   = OneHotEncoder(
    inputCol='sector_idx', outputCol='sector_ohe',
    handleInvalid='keep'
)
 

NUMERIC_FEATURES = [
    'trade_type_encoded',          # always 1 here (BUY) — kept for consistency
    'size_bracket_ordinal',        # 1–8 trade size
    'filed_after_days',            # disclosure speed
    'sigma_30',                    # pre-trade volatility
    'politician_rolling_abn_ret',  # politician's historical AbnRet (anti-leakage)
    'abn_ret_mean',                # blind spot AbnRet
    'abn_ret_max',                 # max anomaly in blind spot
    'anomaly_flag_L1',             # binary: had Level 1 anomaly
    'anomaly_flag_L2',             # binary: had Level 2 anomaly
    'n_blind_spot_days',           # window length
    'committee_ordinal',           # committee proxy
    'owner_encoded',               # who made the trade
    'blind_intensity',             # composite intensity score
    'trade_year',                  # market regime
    'trade_quarter',               # seasonality
    'car_blind_spot',              # CAR-based blind spot metric
    'z_car',                       # Z_CAR across all trades
]
 
CATEGORICAL_FEATURES = [
    'sector_ohe',     # one-hot encoded sector
    'chamber_idx',    # House=0, Senate=1
    'party_idx',      # Democrat=0, Republican=1
]
 
ALL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
 
# Cast numeric features to double
for col_name in NUMERIC_FEATURES:
    sdf = sdf.withColumn(col_name, F.col(col_name).cast(DoubleType()))
 
# Cast Y to double (Spark MLlib requires label as double)
sdf = sdf.withColumn('label', F.col('Y').cast(DoubleType()))
 
# VectorAssembler 
# Combines all features into a single vector column
assembler = VectorAssembler(
    inputCols=ALL_FEATURES,
    outputCol='features',
    handleInvalid='skip'
)
 
print(f"\nFeatures assembled:")
print(f"  Numeric features     : {len(NUMERIC_FEATURES)}")
print(f"  Categorical features : {len(CATEGORICAL_FEATURES)}")
print(f"  Total feature inputs : {len(ALL_FEATURES)}")
 

# CELL 6 — Train / Test Split

In [ ]:
# Stratified split by label
# Spark doesn't have native stratified split so we do it manually
sdf_y1 = sdf.filter(F.col('label') == 1.0)
sdf_y0 = sdf.filter(F.col('label') == 0.0)
 
train_y1, test_y1 = sdf_y1.randomSplit([1-TEST_SPLIT, TEST_SPLIT],
                                        seed=RANDOM_SEED)
train_y0, test_y0 = sdf_y0.randomSplit([1-TEST_SPLIT, TEST_SPLIT],
                                        seed=RANDOM_SEED)
 
train_sdf = train_y1.union(train_y0)
test_sdf  = test_y1.union(test_y0)
 
print(f"Train set : {train_sdf.count():,} rows")
print(f"Test set  : {test_sdf.count():,} rows")
print(f"Train Y=1 : {train_y1.count():,} | Train Y=0 : {train_y0.count():,}")
print(f"Test  Y=1 : {test_y1.count():,}  | Test  Y=0 : {test_y0.count():,}")
 

Train set : 7,663 rows


Test set  : 1,771 rows


Train Y=1 : 3,472 | Train Y=0 : 4,191


Test  Y=1 : 790  | Test  Y=0 : 981


# CELL 7 — Build Pipeline & Train (No CV)

In [ ]:
print("=" * 60)
print("CROSS-VALIDATION HYPERPARAMETER TUNING")
print(f"  Folds : {CV_FOLDS}")
print("=" * 60)
 
# Random Forest classifier
rf = RandomForestClassifier(
    labelCol='label',
    featuresCol='features',
    seed=RANDOM_SEED,
    probabilityCol='probability',
    rawPredictionCol='rawPrediction'
)
 
# Pipeline: index - encode - assemble - classify
pipeline = Pipeline(stages=[
    sector_indexer,
    chamber_indexer,
    party_indexer,
    sector_encoder,
    assembler,
    rf
])
 
# Parameter grid for CV
param_grid = ParamGridBuilder() \
    .addGrid(rf.numTrees,  [100, 200, 300]) \
    .addGrid(rf.maxDepth,  [3, 5, 10]) \
    .build()
 
print(f"Parameter combinations to test: {len(param_grid)}")
print(f"Total models trained (grid × folds): {len(param_grid) * CV_FOLDS}")
 
# AUC-ROC evaluator
evaluator = BinaryClassificationEvaluator(
    labelCol='label',
    metricName='areaUnderROC'
)
 
# Cross validator
cv = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=param_grid,
    evaluator=evaluator,
    numFolds=CV_FOLDS,
    seed=RANDOM_SEED,
    parallelism=2
)
 
print("\nTraining with cross-validation... (this takes 5-15 minutes)")
cv_model = cv.fit(train_sdf)
 
# Best parameters
best_model = cv_model.bestModel
best_rf    = best_model.stages[-1]
 
print(f"\nBest numTrees : {best_rf.getNumTrees}")
print(f"Best maxDepth : {best_rf.getMaxDepth()}")
print(f"Best CV AUC   : {max(cv_model.avgMetrics):.4f}")

CROSS-VALIDATION HYPERPARAMETER TUNING
  Folds : 5
Parameter combinations to test: 9
Total models trained (grid × folds): 45



Training with cross-validation... (this takes 5-15 minutes)


26/04/22 11:11:44 ERROR Instrumentation: org.apache.spark.SparkException: Job 80 cancelled because SparkContext was shut down
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$cleanUpAfterSchedulerStop$1(DAGScheduler.scala:1309)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$cleanUpAfterSchedulerStop$1$adapted(DAGScheduler.scala:1307)
	at scala.collection.mutable.HashSet$Node.foreach(HashSet.scala:450)
	at scala.collection.mutable.HashSet.foreach(HashSet.scala:376)
	at org.apache.spark.scheduler.DAGScheduler.cleanUpAfterSchedulerStop(DAGScheduler.scala:1307)
	at org.apache.spark.scheduler.DAGSchedulerEventProcessLoop.onStop(DAGScheduler.scala:3424)
	at org.apache.spark.util.EventLoop.stop(EventLoop.scala:85)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$stop$3(DAGScheduler.scala:3307)
	at org.apache.spark.util.Utils$.tryLogNonFatalError(Utils.scala:1314)
	at org.apache.spark.scheduler.DAGScheduler.stop(DAGScheduler.scala:3307)
	at org.apache.spark.SparkContext.$anonfun$


Primary CV run failed; retrying with lighter fallback settings...
Reason: Py4JJavaError: An error occurred while calling o6157.fit.
: org.apache.spark.SparkException: Job 80 cancelled because SparkContext was shut down
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$cleanUpAfterSched
Retrying CV with 4 parameter combos × 3 folds...


26/04/22 11:11:45 ERROR TaskContextImpl: Error in TaskCompletionListener
org.apache.spark.SparkException: Block broadcast_115 does not exist
	at org.apache.spark.errors.SparkCoreErrors$.blockDoesNotExistError(SparkCoreErrors.scala:309)
	at org.apache.spark.storage.BlockInfoManager.blockInfo(BlockInfoManager.scala:268)
	at org.apache.spark.storage.BlockInfoManager.unlock(BlockInfoManager.scala:389)
	at org.apache.spark.storage.BlockManager.releaseLock(BlockManager.scala:1349)
	at org.apache.spark.broadcast.TorrentBroadcast.$anonfun$releaseBlockManagerLock$1(TorrentBroadcast.scala:323)
	at org.apache.spark.broadcast.TorrentBroadcast.$anonfun$releaseBlockManagerLock$1$adapted(TorrentBroadcast.scala:323)
	at org.apache.spark.TaskContext$$anon$1.onTaskCompletion(TaskContext.scala:146)
	at org.apache.spark.TaskContextImpl.$anonfun$invokeTaskCompletionListeners$1(TaskContextImpl.scala:153)
	at org.apache.spark.TaskContextImpl.$anonfun$invokeTaskCompletionListeners$1$adapted(TaskContextImpl.sc

ConnectionRefusedError: [Errno 111] Connection refused

# CELL 8 — Evaluate on Test Set

In [ ]:
print("=" * 60)
print("TEST SET EVALUATION")
print("=" * 60)
 
predictions = best_model.transform(test_sdf)
 
# AUC-ROC
test_auc = evaluator.evaluate(predictions)
print(f"\nTest AUC-ROC : {test_auc:.4f}")
 
# Convert to pandas for sklearn metrics
pred_pdf = predictions.select(
    'label', 'prediction',
    F.col('probability').alias('probability')
).toPandas()
 
# Extract probability of class 1
pred_pdf['prob_1'] = pred_pdf['probability'].apply(lambda x: float(x[1]))
 
y_true = pred_pdf['label'].astype(int)
y_pred = pred_pdf['prediction'].astype(int)
y_prob = pred_pdf['prob_1']
 
# Classification report
print(f"\nClassification Report:")
print(classification_report(y_true, y_pred,
                             target_names=['Did not beat SPY', 'Beat SPY']))
 
# Confusion matrix values
cm = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel()
print(f"Confusion Matrix:")
print(f"  True Negative  (TN): {tn:,}  — correctly predicted did not beat SPY")
print(f"  False Positive (FP): {fp:,}  — predicted beat SPY, actually didn't")
print(f"  False Negative (FN): {fn:,}  — predicted didn't beat SPY, actually did")
print(f"  True Positive  (TP): {tp:,}  — correctly predicted beat SPY")
 

# CELL 9 — Feature Importance

In [ ]:
# Get feature names after one-hot encoding
# sector_ohe expands into multiple binary columns
n_sector_categories = (best_model.stages[0]
                        .labels.__len__() 
                        if hasattr(best_model.stages[0], 'labels') 
                        else df['sector'].nunique())
 
# Reconstruct feature names list
sector_ohe_names = [f'sector_{i}' for i in range(n_sector_categories)]
feature_names_expanded = (
    NUMERIC_FEATURES +
    sector_ohe_names +
    ['chamber_idx', 'party_idx']
)
 
importances = best_rf.featureImportances.toArray()
 
# Match importances to feature names
# (expanded list may not perfectly align — use min length)
min_len = min(len(importances), len(feature_names_expanded))
imp_df  = pd.DataFrame({
    'feature':    feature_names_expanded[:min_len],
    'importance': importances[:min_len]
}).sort_values('importance', ascending=False).reset_index(drop=True)
 
imp_df['rank'] = range(1, len(imp_df) + 1)
 
print("=" * 60)
print("FEATURE IMPORTANCES (Gini Impurity Reduction)")
print("=" * 60)
print(imp_df.head(15).to_string(index=False))
 
imp_df.to_csv(IMPORTANCE_OUT, index=False)
print(f"\nSaved to {IMPORTANCE_OUT}")
 
# Key finding: where does committee rank?
committee_rank = imp_df[
    imp_df['feature'] == 'committee_ordinal'
]['rank'].values
 
if len(committee_rank) > 0:
    print(f"\nCommittee assignment rank: #{committee_rank[0]} "
          f"out of {len(imp_df)} features")
    if committee_rank[0] <= 5:
        print("→ Committee ranks in TOP 5 — strong evidence of "
              "domain-specific information advantage")
    elif committee_rank[0] <= 10:
        print("→ Committee ranks in TOP 10 — moderate evidence of "
              "committee-related information advantage")
    else:
        print("→ Committee ranks outside TOP 10 — limited evidence "
              "of committee-specific information advantage")

# CELL 10 — Chart 1: ROC Curve

In [ ]:
fpr, tpr, thresholds = roc_curve(y_true, y_prob)
roc_auc = auc(fpr, tpr)
 
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("Chart 1 — Model Performance",
             fontsize=13, fontweight='bold')
 
# Left: ROC Curve
ax = axes[0]
ax.plot(fpr, tpr, color='#E74C3C', linewidth=2,
        label=f'Random Forest (AUC = {roc_auc:.4f})')
ax.plot([0, 1], [0, 1], color='gray', linestyle='--',
        linewidth=1.2, label='Random baseline (AUC = 0.50)')
ax.fill_between(fpr, tpr, alpha=0.1, color='#E74C3C')
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve")
ax.legend(fontsize=9)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])
 
# Annotate AUC interpretation
if roc_auc >= 0.7:
    interp = "Good discriminative ability"
elif roc_auc >= 0.6:
    interp = "Fair discriminative ability"
else:
    interp = "Limited discriminative ability\n(consistent with efficient market)"
ax.text(0.55, 0.15, interp, fontsize=8,
        bbox=dict(boxstyle='round,pad=0.3',
                  facecolor='lightyellow', alpha=0.8))
 
# Right: Confusion Matrix
ax2 = axes[1]
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['Did not\nbeat SPY', 'Beat\nSPY']
)
disp.plot(ax=ax2, colorbar=False, cmap='Blues')
ax2.set_title("Confusion Matrix (Test Set)")

plt.tight_layout()
out_path = os.path.join('data', 'output', 'chart1_roc_confusion.png')
os.makedirs(os.path.dirname(out_path), exist_ok=True)
plt.savefig(out_path, bbox_inches='tight', dpi=150)
plt.show()
print(f"Chart 1 saved to: {out_path}")

# CELL 11 — Chart 2: Feature Importance

In [ ]:
top_n = 15
top_features = imp_df.head(top_n).copy()
 
# Colour committee feature differently
colors = [
    '#E74C3C' if f == 'committee_ordinal'
    else '#F39C12' if 'sector' in f
    else '#3498DB'
    for f in top_features['feature']
]
 
fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(range(len(top_features)),
               top_features['importance'],
               color=colors, edgecolor='white', linewidth=0.5)
ax.set_yticks(range(len(top_features)))
ax.set_yticklabels(top_features['feature'], fontsize=9)
ax.set_xlabel("Feature Importance (Gini Impurity Reduction)")
ax.set_title(f"Chart 2 — Top {top_n} Feature Importances\n"
             f"Random Forest Classifier",
             fontsize=13, fontweight='bold')
ax.invert_yaxis()
 
# Value labels
for bar in bars:
    ax.text(bar.get_width() + 0.001,
            bar.get_y() + bar.get_height() / 2,
            f"{bar.get_width():.4f}",
            va='center', fontsize=8)
 
# Legend
red_patch    = mpatches.Patch(color='#E74C3C',
                               label='Committee assignment')
orange_patch = mpatches.Patch(color='#F39C12',
                               label='Sector features')
blue_patch   = mpatches.Patch(color='#3498DB',
                               label='Other features')
ax.legend(handles=[red_patch, orange_patch, blue_patch],
          fontsize=9, loc='lower right')
 

plt.tight_layout()
out_path = os.path.join('data', 'output', 'chart2_feature_importance.png')
os.makedirs(os.path.dirname(out_path), exist_ok=True)
plt.savefig(out_path, bbox_inches='tight', dpi=150)
plt.show()
print(f"Chart 2 saved to: {out_path}")

# CELL 12 — Chart 3: Validation AUC Summary

In [ ]:
cv_metrics = cv_model.avgMetrics
param_labels = [
    f"trees={p[rf.numTrees]}, depth={p[rf.maxDepth]}"
    for p in param_grid
]

fig, ax = plt.subplots(figsize=(12, 5))
colors_cv = ['#E74C3C' if v == max(cv_metrics) else '#3498DB'
             for v in cv_metrics]
bars = ax.bar(range(len(cv_metrics)), cv_metrics,
              color=colors_cv, edgecolor='white', linewidth=0.5)
ax.set_xticks(range(len(cv_metrics)))
ax.set_xticklabels(param_labels, rotation=45, ha='right', fontsize=8)
ax.set_ylabel("Validation AUC-ROC")
ax.set_title("Chart 3 — Validation AUC-ROC by Parameter Combination",
             fontsize=13, fontweight='bold')
ax.axhline(0.5, color='gray', linestyle='--',
           linewidth=1.2, label='Random baseline')
ax.set_ylim([0.4, min(1.0, max(cv_metrics) + 0.05)])
ax.legend(fontsize=9)

for bar, val in zip(bars, cv_metrics):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.002,
            f"{val:.3f}", ha='center', va='bottom', fontsize=7)

plt.tight_layout()
out_path = os.path.join('data', 'output', 'chart3_cv_scores.png')
os.makedirs(os.path.dirname(out_path), exist_ok=True)
plt.savefig(out_path, bbox_inches='tight', dpi=150)
plt.show()
print(f"Chart 3 saved to: {out_path}")

# CELL 13 — Save Predictions & Final Summary

In [ ]:
# Save predictions
pred_pdf.to_csv(PREDICTIONS_OUT, index=False)
print(f"Saved {PREDICTIONS_OUT} — {len(pred_pdf):,} test rows")

# AUC interpretation
if test_auc >= 0.7:
    auc_interp = "Good — model has meaningful predictive power"
elif test_auc >= 0.6:
    auc_interp = "Fair — model shows moderate signal above random"
elif test_auc >= 0.55:
    auc_interp = "Weak but above random — consistent with semi-efficient markets"
else:
    auc_interp = "Near random — no meaningful predictive signal detected"

summary_lines = [
    "=" * 60,
    "NOTEBOOK 6 — SUMMARY FOR REPORT",
    "=" * 60,
    "",
    "Task: Predict whether a congressional BUY trade will",
    "      beat the S&P 500 benchmark over 60 trading days.",
    "",
    f"Training samples   : {train_sdf.count():,}",
    f"Test samples       : {test_sdf.count():,}",
    f"Class balance (Y=1): {(buys['Y']==1).mean()*100:.1f}%",
    "",
    "Model Configuration:",
    f"  Algorithm        : Random Forest (Spark MLlib)",
    f"  Tuning method    : Fixed config + internal validation split",
    f"  numTrees         : {best_rf.getNumTrees}",
    f"  maxDepth         : {best_rf.getMaxDepth()}",
    f"  Validation AUC   : {max(cv_model.avgMetrics):.4f}",
    "",
    "Test Set Performance:",
    f"  AUC-ROC          : {test_auc:.4f}",
    f"  Interpretation   : {auc_interp}",
    f"  True Positives   : {tp:,}",
    f"  True Negatives   : {tn:,}",
    f"  False Positives  : {fp:,}",
    f"  False Negatives  : {fn:,}",
    "",
    "Top 5 Most Important Features:",
]

for _, row in imp_df.head(5).iterrows():
    summary_lines.append(
        f"  #{int(row['rank']):<3} {row['feature']:<35} "
        f"{row['importance']:.4f}"
    )

if len(committee_rank) > 0:
    summary_lines += [
        "",
        f"Committee Assignment Rank: #{committee_rank[0]}",
        "Committee finding: " + (
            "SIGNIFICANT — committee assignment is a top predictor, "
            "consistent with domain-specific information advantage."
            if committee_rank[0] <= 10
            else "LIMITED — committee assignment does not rank as a "
            "top predictor. Other factors drive congressional trade outcomes."
        ),
    ]

summary_lines += [
    "",
    "Limitations:",
    "  - Committee data approximated from public records (not from dataset)",
    "  - Restricted to BUY trades only (sell trades excluded)",
    "  - Y label based on 60-day holding period (arbitrary window)",
    "  - AUC interpretation must account for market efficiency",
    "",
    "Notebook 6 complete. All notebooks finished.",
]

summary_text = "\n".join(summary_lines)
print(summary_text)

with open(SUMMARY_OUT, 'w') as f:
    f.write(summary_text)
print(f"\nSaved {SUMMARY_OUT}")

spark.stop()